# GeoCPU：今の評価を読み取るライブモニター

**別のCPU Colabセッションで開いてください。** 元のGPUセッションの実行中セルはそのまま継続します。同じカーネルで新しいセルを実行すると、現在の評価が終わるまで待ちになるためです。

保存された原回答・採点・ログを5秒ごとに読み取り、処理件数と実際の正答数を分けて表示します。元の実験へ書き込みません。Driveの同期遅延分は表示に遅れが出ます。ここでの停止ボタンはこの監視だけを止めます。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, os, stat, sys, tempfile, zipfile

STUDY = Path('/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106')
ARCHIVE = STUDY / 'GeoCPU_practical_training_logging_20261107' / 'GeoCPU_practical_training_logging_20261107.zip'
if not ARCHIVE.is_file():
    raise FileNotFoundError(f'成果物ZIPの場所をARCHIVEへ指定してください: {ARCHIVE}')

# 既存コードへ重ねず、毎回新しいローカル作業フォルダーへ展開します。
WORK = Path(tempfile.mkdtemp(prefix='geocpu_practical_', dir='/content'))
with zipfile.ZipFile(ARCHIVE) as z:
    names=z.namelist()
    if len(names)!=len(set(names)):
        raise ValueError('ZIPの重複パス')
    for m in z.infolist():
        if not (WORK/m.filename).resolve().is_relative_to(WORK.resolve()) or stat.S_ISLNK(m.external_attr >> 16):
            raise ValueError('ZIPの不正なパス')
    manifest=json.loads(z.read('PACKAGE_MANIFEST.json'))
    expected={r['path']:r for r in manifest['files']}
    actual={m.filename for m in z.infolist() if not m.is_dir()}-{'PACKAGE_MANIFEST.json'}
    if actual!=set(expected):
        raise ValueError('ZIPと固定manifestのファイル集合が不一致')
    for name, row in expected.items():
        raw=z.read(name)
        if len(raw)!=row['size_bytes'] or hashlib.sha256(raw).hexdigest()!=row['sha256']:
            raise ValueError(f'内容ハッシュ不一致: {name}')
    z.extractall(WORK)
sys.path.insert(0, str(WORK))
os.chdir(WORK)
print('作業フォルダー:', WORK)
print('固定ファイル検証:', len(expected), '件')


## 対象を選び、表示を開始

初期値は現在のstep30,500評価です。新しい評価を見たい場合は `WATCH` をその結果フォルダーへ変えてください。処理済み40/120のような値は、40問正解という意味ではありません。


In [ ]:
from live_logging import watch_existing
WATCH=STUDY/'v47_train_eval_suite_step_30500'
watch_existing(WATCH,poll_s=5,notebook=True)


## 一度だけ確認する場合

上の監視セルを停止してから次のセルを実行します。正答数、分母、未計測数、エラー数、最終更新を確認できます。


In [ ]:
watch_existing(WATCH,once=True,notebook=True)
